# Series 2 — CDC, AUTO CDC, SCD Type 2, and Dimensional Modeling

## Objective

Use a real Instacart product catalog as the baseline and simulate realistic source-system changes.

This project will compare:

- Delta Change Data Feed
- SCD Type 1 current-state processing
- SCD Type 2 historical processing
- Lakeflow AUTO CDC
- Point-in-time fact-to-dimension joins

## Important note

Instacart provides historical snapshot files, not a live CDC feed.

The baseline data is real, but the product changes are intentionally simulated so that INSERT, UPDATE, DELETE, sequencing, and historical tracking can be tested clearly.

In [0]:
%sql

USE CATALOG workspace;
USE SCHEMA instacart_lab;

In [0]:
raw_path = "/Volumes/workspace/instacart_lab/instacart_volume/raw"

display(dbutils.fs.ls(raw_path))

In [0]:
products_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "false")
    .csv(f"{raw_path}/products.csv")
)

aisles_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "false")    
    .csv(f"{raw_path}/aisles.csv")
)

departments_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "false")
    .csv(f"{raw_path}/departments.csv")
)

source_counts = [
    ("products", products_df.count()),
    ("aisles", aisles_df.count()),
    ("departments", departments_df.count())
]

display(
    spark.createDataFrame(
        source_counts,
        ["source_name", "record_count"]
    )
)

In [0]:
products_df.createOrReplaceTempView("products_raw")
aisles_df.createOrReplaceTempView("aisles_raw")
departments_df.createOrReplaceTempView("departments_raw")

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.instacart_lab.product_master
TBLPROPERTIES (
  delta.enableChangeDataFeed = true
)
AS
SELECT
    p.product_id,
    p.product_name,
    p.aisle_id,
    a.aisle,
    p.department_id,
    d.department,
    true AS is_active,
    current_timestamp() AS created_at,
    current_timestamp() AS updated_at
FROM products_raw p
LEFT JOIN aisles_raw a
    ON p.aisle_id = a.aisle_id
LEFT JOIN departments_raw d
    ON p.department_id = d.department_id;

In [0]:
%sql

SHOW VIEWS IN workspace.instacart_lab;

In [0]:
table_name = "workspace.instacart_lab.product_master"

source_count = products_df.count()
target_count = spark.table(table_name).count()

assert target_count == source_count, (
    f"Expected {source_count:,} products, "
    f"but product_master contains {target_count:,}"
)

cdf_property = (
    spark.sql(f"SHOW TBLPROPERTIES {table_name}")
    .filter("key = 'delta.enableChangeDataFeed'")
    .select("value")
    .first()
)

assert cdf_property is not None, "CDF property was not found"
assert cdf_property["value"] == "true", "CDF is not enabled"

duplicate_products = (
    spark.table(table_name)
    .groupBy("product_id")
    .count()
    .filter("count > 1")
    .count()
)

assert duplicate_products == 0, (
    f"Found {duplicate_products} duplicate product IDs"
)

print("✅ Baseline product table created")
print(f"✅ Product rows: {target_count:,}")
print("✅ Change Data Feed enabled")
print("✅ Product business key is unique")

In [0]:
%sql

DESCRIBE HISTORY workspace.instacart_lab.product_master;

In [0]:
from pyspark.sql import functions as F

table_name = "workspace.instacart_lab.product_master"

# Remove products inserted by an earlier demo run.
spark.sql(f"""
DELETE FROM {table_name}
WHERE product_id IN (999998, 999999)
""")

# Restore products 1, 2 and 3 from the original source views.
spark.sql(f"""
MERGE INTO {table_name} AS target
USING (
    SELECT
        p.product_id,
        p.product_name,
        p.aisle_id,
        a.aisle,
        p.department_id,
        d.department,
        true AS is_active
    FROM products_raw p
    LEFT JOIN aisles_raw a
        ON p.aisle_id = a.aisle_id
    LEFT JOIN departments_raw d
        ON p.department_id = d.department_id
    WHERE p.product_id IN (1, 2, 3)
) AS source
ON target.product_id = source.product_id

WHEN MATCHED THEN UPDATE SET
    target.product_name = source.product_name,
    target.aisle_id = source.aisle_id,
    target.aisle = source.aisle,
    target.department_id = source.department_id,
    target.department = source.department,
    target.is_active = true,
    target.updated_at = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
    product_id,
    product_name,
    aisle_id,
    aisle,
    department_id,
    department,
    is_active,
    created_at,
    updated_at
)
VALUES (
    source.product_id,
    source.product_name,
    source.aisle_id,
    source.aisle,
    source.department_id,
    source.department,
    source.is_active,
    current_timestamp(),
    current_timestamp()
)
""")

start_version = (
    spark.sql(f"DESCRIBE HISTORY {table_name}")
    .agg(F.max("version").alias("version"))
    .first()["version"]
)

print(f"Starting version for this CDC test: {start_version}")

In [0]:
%sql

INSERT INTO workspace.instacart_lab.product_master
SELECT
    999999 AS product_id,
    'New Organic Snack Bar' AS product_name,
    p.aisle_id,
    a.aisle,
    p.department_id,
    d.department,
    true AS is_active,
    current_timestamp() AS created_at,
    current_timestamp() AS updated_at
FROM products_raw p
LEFT JOIN aisles_raw a
    ON p.aisle_id = a.aisle_id
LEFT JOIN departments_raw d
    ON p.department_id = d.department_id
WHERE p.product_id = 4;

In [0]:
%sql
UPDATE workspace.instacart_lab.product_master
SET
    product_name = CONCAT(product_name, ' - Updated Packaging'),
    updated_at = current_timestamp()
WHERE product_id = 1;

In [0]:
%sql
DELETE FROM workspace.instacart_lab.product_master
WHERE product_id = 2;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW product_merge_source AS

-- Existing product: update product 3
SELECT
    3 AS product_id,
    CONCAT(target.product_name, ' - Reformulated') AS product_name,
    ref.aisle_id,
    a.aisle,
    ref.department_id,
    d.department,
    true AS is_active
FROM workspace.instacart_lab.product_master target
CROSS JOIN products_raw ref
LEFT JOIN aisles_raw a
    ON ref.aisle_id = a.aisle_id
LEFT JOIN departments_raw d
    ON ref.department_id = d.department_id
WHERE target.product_id = 3
  AND ref.product_id = 5

UNION ALL

-- New product: insert product 999998
SELECT
    999998 AS product_id,
    'New Plant-Based Breakfast Bowl' AS product_name,
    ref.aisle_id,
    a.aisle,
    ref.department_id,
    d.department,
    true AS is_active
FROM products_raw ref
LEFT JOIN aisles_raw a
    ON ref.aisle_id = a.aisle_id
LEFT JOIN departments_raw d
    ON ref.department_id = d.department_id
WHERE ref.product_id = 6;

In [0]:
%sql

MERGE INTO workspace.instacart_lab.product_master AS target
USING product_merge_source AS source
ON target.product_id = source.product_id

WHEN MATCHED THEN UPDATE SET
    target.product_name = source.product_name,
    target.aisle_id = source.aisle_id,
    target.aisle = source.aisle,
    target.department_id = source.department_id,
    target.department = source.department,
    target.is_active = source.is_active,
    target.updated_at = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
    product_id,
    product_name,
    aisle_id,
    aisle,
    department_id,
    department,
    is_active,
    created_at,
    updated_at
)
VALUES (
    source.product_id,
    source.product_name,
    source.aisle_id,
    source.aisle,
    source.department_id,
    source.department,
    source.is_active,
    current_timestamp(),
    current_timestamp()
);

In [0]:
end_version = (
    spark.sql(f"DESCRIBE HISTORY {table_name}")
    .agg(F.max("version").alias("version"))
    .first()["version"]
)

history_df = (
    spark.sql(f"DESCRIBE HISTORY {table_name}")
    .select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics"
    )
    .filter(F.col("version") > start_version)
    .orderBy("version")
)

print(f"Starting version: {start_version}")
print(f"Ending version:   {end_version}")

display(history_df)

In [0]:
cdf_start_version = start_version + 1

cdf_df = spark.sql(f"""
SELECT
    product_id,
    product_name,
    aisle,
    department,
    is_active,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes(
    '{table_name}',
    {cdf_start_version},
    {end_version}
)
ORDER BY
    _commit_version,
    product_id,
    CASE _change_type
        WHEN 'update_preimage' THEN 1
        WHEN 'update_postimage' THEN 2
        WHEN 'insert' THEN 3
        WHEN 'delete' THEN 4
        ELSE 5
    END
""")

display(cdf_df)

In [0]:
cdf_summary_df = (
    cdf_df
    .groupBy("_commit_version", "_change_type")
    .count()
    .orderBy("_commit_version", "_change_type")
)

display(cdf_summary_df)

In [0]:
display(
    cdf_df
    .groupBy("_change_type")
    .count()
    .orderBy("_change_type")
)

In [0]:
change_counts = {
    row["_change_type"]: row["count"]
    for row in cdf_df.groupBy("_change_type").count().collect()
}

expected_counts = {
    "insert": 2,
    "update_preimage": 2,
    "update_postimage": 2,
    "delete": 1
}

assert cdf_df.count() == 7, (
    f"Expected 7 CDF rows, found {cdf_df.count()}"
)

for change_type, expected_count in expected_counts.items():
    actual_count = change_counts.get(change_type, 0)

    assert actual_count == expected_count, (
        f"Expected {expected_count} {change_type} rows, "
        f"but found {actual_count}"
    )

current_df = spark.table(table_name)

assert current_df.filter("product_id = 1").count() == 1
assert current_df.filter("product_id = 2").count() == 0
assert current_df.filter("product_id = 3").count() == 1
assert current_df.filter("product_id = 999998").count() == 1
assert current_df.filter("product_id = 999999").count() == 1

assert (
    current_df
    .filter("product_id = 1")
    .filter("product_name LIKE '%Updated Packaging%'")
    .count()
    == 1
), "Product 1 update was not applied"

assert (
    current_df
    .filter("product_id = 3")
    .filter("product_name LIKE '%Reformulated%'")
    .count()
    == 1
), "Product 3 MERGE update was not applied"

print("✅ INSERT captured by CDF")
print("✅ UPDATE preimage and postimage captured")
print("✅ DELETE captured by CDF")
print("✅ MERGE update and insert captured")
print("✅ Current product table reflects all changes")
print("✅ CDF validation passed: 7 change records")

In [0]:
%sql

SELECT
    _change_type,
    COUNT(*) AS change_count
FROM table_changes(
    'workspace.instacart_lab.product_master',
    0
)
GROUP BY _change_type
ORDER BY _change_type;

In [0]:
%sql
SELECT
    product_id,
    product_name,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes(
    'workspace.instacart_lab.product_master',
    0
)
WHERE product_id IN (1, 2, 3, 999998, 999999)
ORDER BY
    _commit_version,
    product_id,
    _change_type;

In [0]:
%sql

SELECT
    product_id,
    product_name,
    aisle,
    department,
    is_active,
    created_at,
    updated_at
FROM workspace.instacart_lab.dim_product_scd1
WHERE product_id IN (1, 2, 3, 999998, 999999)
ORDER BY product_id;

In [0]:
%sql

SELECT
    product_id,
    product_name,
    aisle,
    department,
    is_active,
    __START_AT,
    __END_AT
FROM workspace.instacart_lab.dim_product_scd2
WHERE product_id IN (1, 2, 3, 999998, 999999)
ORDER BY
    product_id,
    __START_AT;

In [0]:
%sql

SELECT
    (SELECT COUNT(*)
     FROM workspace.instacart_lab.dim_product_scd1)
        AS scd1_current_count,

    (SELECT COUNT(*)
     FROM workspace.instacart_lab.dim_product_scd2
     WHERE __END_AT IS NULL)
        AS scd2_current_count;

In [0]:
%sql

SELECT
    product_id,
    COUNT(*) AS current_version_count
FROM workspace.instacart_lab.dim_product_scd2
WHERE __END_AT IS NULL
GROUP BY product_id
HAVING COUNT(*) > 1;

In [0]:
scd1_table = "workspace.instacart_lab.dim_product_scd1"
scd2_table = "workspace.instacart_lab.dim_product_scd2"

scd1_df = spark.table(scd1_table)
scd2_df = spark.table(scd2_table)

scd1_count = scd1_df.count()

scd2_current_df = scd2_df.filter("__END_AT IS NULL")
scd2_current_count = scd2_current_df.count()

duplicate_current_keys = (
    scd2_current_df
    .groupBy("product_id")
    .count()
    .filter("count > 1")
    .count()
)

null_scd1_keys = (
    scd1_df
    .filter("product_id IS NULL")
    .count()
)

null_scd2_keys = (
    scd2_df
    .filter("product_id IS NULL")
    .count()
)

assert scd1_count == scd2_current_count, (
    f"SCD1 contains {scd1_count:,} current rows, "
    f"but current SCD2 contains {scd2_current_count:,}"
)

assert duplicate_current_keys == 0, (
    f"Found {duplicate_current_keys} products with "
    "multiple current SCD2 versions"
)

assert null_scd1_keys == 0, (
    f"SCD1 contains {null_scd1_keys} null product keys"
)

assert null_scd2_keys == 0, (
    f"SCD2 contains {null_scd2_keys} null product keys"
)

assert scd1_df.filter("product_id = 2").count() == 0, (
    "Deleted product 2 still exists in SCD1"
)

assert (
    scd2_df
    .filter("product_id = 2 AND __END_AT IS NULL")
    .count()
    == 0
), "Deleted product 2 still has a current SCD2 version"

assert (
    scd2_df
    .filter("product_id = 1")
    .count()
    >= 2
), "Updated product 1 does not have SCD2 history"

assert (
    scd2_df
    .filter("product_id = 3")
    .count()
    >= 2
), "Updated product 3 does not have SCD2 history"

print("✅ SCD1 and current SCD2 counts reconcile")
print("✅ No duplicate current SCD2 versions")
print("✅ No null product business keys")
print("✅ DELETE was correctly applied")
print("✅ Product 1 history was preserved")
print("✅ Product 3 history was preserved")
print("✅ AUTO CDC SCD1 and SCD2 validation passed")

In [0]:
raw_path = "/Volumes/workspace/instacart_lab/instacart_volume/raw"

orders_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "false")
    .csv(f"{raw_path}/orders.csv")
)

order_products_train_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "false")
    .csv(f"{raw_path}/order_products__train.csv")
)

orders_df.createOrReplaceTempView("orders_raw")
order_products_train_df.createOrReplaceTempView(
    "order_products_train_raw"
)

print(f"Orders: {orders_df.count():,}")
print(f"Order items: {order_products_train_df.count():,}")

In [0]:
%sql

SELECT
    product_id,
    COUNT(*) AS order_item_count,
    COUNT(DISTINCT order_id) AS order_count
FROM order_products_train_raw
WHERE product_id IN (1, 2, 3)
GROUP BY product_id
ORDER BY product_id;

In [0]:
%sql

CREATE OR REPLACE TABLE
workspace.instacart_lab.fact_order_items_base AS

SELECT
    sha2(
        concat_ws(
            '||',
            CAST(op.order_id AS STRING),
            CAST(op.product_id AS STRING)
        ),
        256
    ) AS order_item_key,

    op.order_id,
    o.user_id,
    op.product_id,

    o.eval_set,
    o.order_number,
    o.order_dow,
    o.order_hour_of_day,
    o.days_since_prior_order,

    op.add_to_cart_order,
    op.reordered

FROM order_products_train_raw op

INNER JOIN orders_raw o
    ON op.order_id = o.order_id;

In [0]:
%sql

SELECT *
FROM workspace.instacart_lab.fact_order_items_base
LIMIT 20;

In [0]:
fact_base = spark.table(
    "workspace.instacart_lab.fact_order_items_base"
)

duplicate_grain_count = (
    fact_base
    .groupBy("order_id", "product_id")
    .count()
    .filter("count > 1")
    .count()
)

null_key_count = (
    fact_base
    .filter("order_id IS NULL OR product_id IS NULL")
    .count()
)

assert duplicate_grain_count == 0, (
    f"Found {duplicate_grain_count} duplicate "
    "(order_id, product_id) fact grains"
)

assert null_key_count == 0, (
    f"Found {null_key_count} fact rows with null keys"
)

print("✅ Fact grain: one row per product within an order")
print(f"✅ Fact rows: {fact_base.count():,}")
print("✅ No duplicate fact grains")
print("✅ No null business keys")

In [0]:
%sql

CREATE OR REPLACE TABLE
workspace.instacart_lab.dim_product AS

-- Unknown member
SELECT
    sha2('UNKNOWN_PRODUCT', 256) AS product_sk,
    -1 AS product_id,
    'Unknown Product' AS product_name,
    -1 AS aisle_id,
    'Unknown Aisle' AS aisle,
    -1 AS department_id,
    'Unknown Department' AS department,
    false AS is_active,
    TIMESTAMP('1900-01-01 00:00:00') AS effective_start_ts,
    CAST(NULL AS TIMESTAMP) AS effective_end_ts,
    true AS is_current

UNION ALL

-- Historical SCD2 records produced by AUTO CDC
SELECT
    sha2(
        concat_ws(
            '||',
            CAST(product_id AS STRING),
            CAST(__START_AT AS STRING)
        ),
        256
    ) AS product_sk,

    product_id,
    product_name,
    aisle_id,
    aisle,
    department_id,
    department,
    is_active,

    __START_AT AS effective_start_ts,
    __END_AT AS effective_end_ts,

    CASE
        WHEN __END_AT IS NULL THEN true
        ELSE false
    END AS is_current

FROM workspace.instacart_lab.dim_product_scd2;

In [0]:
%sql

SELECT
    product_sk,
    product_id,
    product_name,
    aisle,
    department,
    effective_start_ts,
    effective_end_ts,
    is_current
FROM workspace.instacart_lab.dim_product
WHERE product_id IN (1, 2, 3)
ORDER BY product_id, effective_start_ts;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW product_change_boundaries AS

WITH version_summary AS (
    SELECT
        product_id,
        COUNT(*) AS version_count,
        MIN(effective_start_ts) AS first_start_ts,
        MAX(effective_start_ts) AS latest_start_ts,
        MAX(effective_end_ts) AS latest_end_ts
    FROM workspace.instacart_lab.dim_product
    WHERE product_id IN (1, 2, 3)
    GROUP BY product_id
)

SELECT
    product_id,

    CASE
        WHEN version_count > 1
            THEN latest_start_ts
        ELSE latest_end_ts
    END AS change_boundary_ts,

    version_count

FROM version_summary;

In [0]:
%sql

SELECT *
FROM product_change_boundaries
ORDER BY product_id;

In [0]:
%sql

CREATE OR REPLACE TABLE
workspace.instacart_lab.fact_order_items_pti_test AS

WITH selected_real_order_items AS (
    SELECT
        order_item_key,
        order_id,
        user_id,
        product_id,
        eval_set,
        order_number,
        order_dow,
        order_hour_of_day,
        days_since_prior_order,
        add_to_cart_order,
        reordered

    FROM workspace.instacart_lab.fact_order_items_base

    WHERE product_id IN (1, 2, 3)

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY product_id
        ORDER BY order_id
    ) = 1
),

scenarios AS (
    SELECT *
    FROM VALUES
        ('BEFORE_CHANGE', -1),
        ('AFTER_CHANGE', 1)
    AS scenario_values(change_scenario, minute_offset)
)

SELECT
    sha2(
        concat_ws(
            '||',
            real_order.order_item_key,
            scenario.change_scenario
        ),
        256
    ) AS fact_test_key,

    real_order.order_item_key,
    real_order.order_id,
    real_order.user_id,
    real_order.product_id,

    real_order.eval_set,
    real_order.order_number,
    real_order.order_dow,
    real_order.order_hour_of_day,
    real_order.days_since_prior_order,
    real_order.add_to_cart_order,
    real_order.reordered,

    scenario.change_scenario,

    CASE
        WHEN scenario.minute_offset = -1
            THEN boundary.change_boundary_ts - INTERVAL 1 MINUTE

        WHEN scenario.minute_offset = 1
            THEN boundary.change_boundary_ts + INTERVAL 1 MINUTE
    END AS order_event_ts,

    boundary.change_boundary_ts

FROM selected_real_order_items real_order

INNER JOIN product_change_boundaries boundary
    ON real_order.product_id = boundary.product_id

CROSS JOIN scenarios scenario;

In [0]:
%sql

SELECT
    product_id,
    order_id,
    change_scenario,
    order_event_ts,
    change_boundary_ts
FROM workspace.instacart_lab.fact_order_items_pti_test
ORDER BY product_id, order_event_ts;

In [0]:
%sql

CREATE OR REPLACE TABLE
workspace.instacart_lab.fact_order_items_pti_enriched AS

WITH historical_match AS (
    SELECT
        fact.*,

        dimension.product_sk AS matched_product_sk,
        dimension.product_name AS product_name_at_order,
        dimension.aisle AS aisle_at_order,
        dimension.department AS department_at_order,
        dimension.effective_start_ts AS matched_dimension_start_ts,
        dimension.effective_end_ts AS matched_dimension_end_ts

    FROM workspace.instacart_lab.fact_order_items_pti_test fact

    LEFT JOIN workspace.instacart_lab.dim_product dimension
        ON fact.product_id = dimension.product_id

       AND fact.order_event_ts >=
           dimension.effective_start_ts

       AND (
           dimension.effective_end_ts IS NULL
           OR fact.order_event_ts <
              dimension.effective_end_ts
       )
)

SELECT
    fact_test_key,
    order_item_key,
    order_id,
    user_id,
    product_id,

    eval_set,
    order_number,
    order_dow,
    order_hour_of_day,
    days_since_prior_order,
    add_to_cart_order,
    reordered,

    change_scenario,
    order_event_ts,
    change_boundary_ts,

    COALESCE(
        matched_product_sk,
        sha2('UNKNOWN_PRODUCT', 256)
    ) AS product_sk,

    COALESCE(
        product_name_at_order,
        'Unknown Product'
    ) AS product_name_at_order,

    COALESCE(
        aisle_at_order,
        'Unknown Aisle'
    ) AS aisle_at_order,

    COALESCE(
        department_at_order,
        'Unknown Department'
    ) AS department_at_order,

    matched_dimension_start_ts,
    matched_dimension_end_ts,

    CASE
        WHEN matched_product_sk IS NULL
            THEN 'UNKNOWN_DIMENSION_MEMBER'
        ELSE 'HISTORICAL_VERSION_MATCHED'
    END AS dimension_match_status

FROM historical_match;

In [0]:
%sql

SELECT
    product_id,
    change_scenario,
    order_event_ts,
    product_name_at_order,
    aisle_at_order,
    department_at_order,
    matched_dimension_start_ts,
    matched_dimension_end_ts,
    dimension_match_status
FROM workspace.instacart_lab.fact_order_items_pti_enriched
ORDER BY product_id, order_event_ts;

In [0]:
%sql

SELECT
    fact.product_id,
    fact.change_scenario,
    fact.order_event_ts,

    scd1.product_name AS scd1_current_product_name,

    fact.product_name_at_order
        AS scd2_historical_product_name,

    fact.dimension_match_status

FROM workspace.instacart_lab.fact_order_items_pti_enriched fact

LEFT JOIN workspace.instacart_lab.dim_product_scd1 scd1
    ON fact.product_id = scd1.product_id

ORDER BY
    fact.product_id,
    fact.order_event_ts;

In [0]:

from pyspark.sql import functions as F

dimension_table = "workspace.instacart_lab.dim_product"
fact_table = "workspace.instacart_lab.fact_order_items_pti_enriched"

dimension_df = spark.table(dimension_table)
fact_df = spark.table(fact_table)

# Surrogate keys must be unique.
duplicate_surrogate_keys = (
    dimension_df
    .groupBy("product_sk")
    .count()
    .filter("count > 1")
    .count()
)

# A business key can have only one current version.
duplicate_current_products = (
    dimension_df
    .filter("is_current = true AND product_id <> -1")
    .groupBy("product_id")
    .count()
    .filter("count > 1")
    .count()
)

# Every fact must contain a usable foreign key.
null_product_foreign_keys = (
    fact_df
    .filter("product_sk IS NULL")
    .count()
)

# Every fact foreign key must exist in the dimension.
orphan_foreign_keys = (
    fact_df
    .select("product_sk")
    .distinct()
    .join(
        dimension_df.select("product_sk").distinct(),
        on="product_sk",
        how="left_anti"
    )
    .count()
)

assert duplicate_surrogate_keys == 0, (
    f"Found {duplicate_surrogate_keys} duplicate surrogate keys"
)

assert duplicate_current_products == 0, (
    f"Found {duplicate_current_products} products "
    "with multiple current versions"
)

assert null_product_foreign_keys == 0, (
    f"Found {null_product_foreign_keys} facts "
    "with null product foreign keys"
)

assert orphan_foreign_keys == 0, (
    f"Found {orphan_foreign_keys} fact foreign keys "
    "missing from dim_product"
)

# Updated products must map to different historical keys.
for product_id in [1, 3]:
    distinct_keys = (
        fact_df
        .filter(F.col("product_id") == product_id)
        .select("product_sk")
        .distinct()
        .count()
    )

    assert distinct_keys == 2, (
        f"Product {product_id} should map to two "
        f"historical product keys, found {distinct_keys}"
    )

# Deleted product must match history before deletion.
assert (
    fact_df
    .filter(
        """
        product_id = 2
        AND change_scenario = 'BEFORE_CHANGE'
        AND dimension_match_status =
            'HISTORICAL_VERSION_MATCHED'
        """
    )
    .count()
    == 1
), "Product 2 did not match its pre-deletion history"

# After deletion, the product should map to unknown.
assert (
    fact_df
    .filter(
        """
        product_id = 2
        AND change_scenario = 'AFTER_CHANGE'
        AND dimension_match_status =
            'UNKNOWN_DIMENSION_MEMBER'
        """
    )
    .count()
    == 1
), "Product 2 after deletion did not map to unknown"

print("✅ Dimension surrogate keys are unique")
print("✅ One current version per product")
print("✅ Fact foreign keys are populated")
print("✅ All fact foreign keys exist in the dimension")
print("✅ Products 1 and 3 use different historical versions")
print("✅ Product 2 matches history before deletion")
print("✅ Product 2 maps to unknown after deletion")
print("✅ Point-in-time dimensional model validation passed")